# 🛣️ RDD2022 Pothole Dataset Preprocessing for YOLOv8 (v2.3)

This version uses a **Functional Workflow**. Each step is encapsulated in a separate Python function to maintain a clean, modular, and research-grade pipeline.

## 1. Pipeline Definitions

In [ ]:
from google.colab import drive
import os, sys, shutil
from pathlib import Path

# Global Configuration
PROJECT_NAME = "pothole_project"
DRIVE_ROOT = Path(f"/content/drive/MyDrive/{PROJECT_NAME}")
SCRIPTS_DIR = DRIVE_ROOT / "notebooks"
DRIVE_PROCESSED = DRIVE_ROOT / "processed_dataset"

LOCAL_RAW = Path("/content/raw_data")
LOCAL_WORK = Path("/content/workspace")
LOCAL_OUT = Path("/content/yolo_dataset")

def setup_environment():
    """Mounts Drive and prepares local ephemeral folders."""
    drive.mount('/content/drive')
    
    # Create Drive structure
    for p in [DRIVE_ROOT / 'raw_dataset', DRIVE_PROCESSED, SCRIPTS_DIR, DRIVE_ROOT / 'models']:
        p.mkdir(parents=True, exist_ok=True)

    # Prepare local folders
    for p in [LOCAL_RAW, LOCAL_WORK, LOCAL_OUT]:
        if p.exists(): shutil.rmtree(p)
        p.mkdir(parents=True)
    
    # Prioritize local scripts in sys.path
    if str(SCRIPTS_DIR) not in sys.path:
        sys.path.insert(0, str(SCRIPTS_DIR))
    
    print("✅ Environment setup complete.")

def download_data(username, key, dataset_slug):
    """Uses existing utils.downloader logic to acquire the dataset."""
    os.environ['KAGGLE_USERNAME'] = username
    os.environ['KAGGLE_KEY'] = key
    
    import utils.downloader
    print(f"⏬ Downloading {dataset_slug}...")
    success = utils.downloader.download_dataset(dataset_slug, LOCAL_RAW)
    if success:
        print("✅ Download successful.")
        !ls -R {str(LOCAL_RAW)} | head -n 10
    else:
        print("❌ Download failed.")

def run_preprocessing_pipeline():
    """Executes the modular preprocessing script."""
    # Configure environment for config.py
    os.environ['RDD2022_ROOT'] = str(LOCAL_RAW / "rdd2022") # Update folder name if needed
    os.environ['WORK_DIR'] = str(LOCAL_WORK)
    os.environ['YOLO_DATASET_ROOT'] = str(LOCAL_OUT)
    os.environ['LOG_DIR'] = str(LOCAL_WORK / "logs")
    
    print("🚀 Executing preprocess_rdd2022.main()...")
    try:
        import preprocess_rdd2022
        preprocess_rdd2022.main()
        print("✅ Processing complete.")
    except Exception as e:
        print(f"❌ Error: {e}")

def save_to_drive():
    """Generates data.yaml and syncs final data to Drive."""
    # 1. YAML Generation
    yaml_content = f"path: {str(DRIVE_PROCESSED)}\ntrain: images/train\nval: images/val\ntest: images/test\n\nnames:\n  0: Pothole"
    with open(LOCAL_OUT / "data.yaml", "w") as f:
        f.write(yaml_content)
    
    # 2. Syncing
    print(f"📤 Syncing to Drive: {DRIVE_PROCESSED}...")
    !rsync -ah --progress {str(LOCAL_OUT)}/ {str(DRIVE_PROCESSED)}/
    print("✅ Persistence complete.")

## 2. Execution
Run each function in sequence to execute the pipeline.

In [ ]:
# Step 1: Install & Mount
!pip install -q ultralytics imagehash python-dotenv kaggle opencv-python
setup_environment()

In [ ]:
# Step 2: Download
download_data(
    username="your_username", 
    key="your_api_key", 
    dataset_slug="aliabdelmenam/rdd-2022"
)

In [ ]:
# Step 3: Preprocess
run_preprocessing_pipeline()

In [ ]:
# Step 4: Save
save_to_drive()